# Resumen automático — Extractivo vs Abstractivo con Transformers (**SOLVED**)

**Task:** Automatic text summarization. Compare a simple **extractive** method with a **Transformer-based abstractive** model.

**Models / tools:** 
- `sshleifer/distilbart-cnn-12-6` (distilled BART, abstractive summarization).
- Implementación propia de resumen extractivo (frecuencia de palabras + selección de frases).

---

Esta es la **versión resuelta** del notebook. Incluye todas las celdas de código completas y comentadas.
Se recomienda intentar primero la versión **STUDENT** y usar esta solo como referencia o para corregir.

---

## Objetivos del notebook

En este notebook vas a:

1. Cargar un modelo de resumen **abstractivo** preentrenado usando `transformers`.
2. Implementar un pequeño sistema de resumen **extractivo** basado en puntuación de frases.
3. Comparar ambos enfoques (extractivo vs abstractivo) sobre varios textos.
4. Implementar una mini-métrica basada en **solapamiento de palabras** (aprox. tipo ROUGE-1 muy simplificado).
5. Analizar ventajas, limitaciones y errores típicos de cada enfoque.


## 1) (Opcional) Instalación / actualización de librerías

In [ ]:
# (Opcional) Instalación / actualización de librerías en Colab.
# Normalmente bastará con ejecutar esta celda una vez al principio del notebook.

# import sys
# pip = f"{sys.executable} -m pip"
# !{pip} install -q "transformers>=4.40.0" datasets nltk

## 2) Imports y configuración básica

In [ ]:
from transformers import pipeline
import numpy as np
import re
import nltk

# Descargamos modelos de segmentación de frases de NLTK
nltk.download("punkt")

print("Librerías importadas correctamente.")

## 3) Carga del modelo de resumen abstractivo (Transformers)

Usamos `sshleifer/distilbart-cnn-12-6`, un modelo distilado de BART entrenado para resumen de noticias en inglés.


In [ ]:
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

# Prueba rápida con un texto sencillo
test_text = (
    "Natural language processing (NLP) is a subfield of artificial intelligence "
    "that focuses on the interaction between computers and humans through natural language. "
    "Most NLP techniques rely on machine learning to derive meaning from human languages."
)

summary = summarizer(test_text, max_length=60, min_length=20, num_beams=4)[0]["summary_text"]

print("=== Texto original ===")
print(test_text)
print("\n=== Resumen (abstractivo) ===")
print(summary)

## 4) Definición de un pequeño conjunto de documentos

In [ ]:
documents = [
    (
        "Natural language processing (NLP) is a subfield of artificial intelligence that focuses "
        "on the interaction between computers and humans through natural language. The ultimate "
        "objective of NLP is to read, decipher, understand, and make sense of human language in a "
        "valuable way. Most NLP techniques rely on machine learning to derive meaning from human languages."
    ),
    (
        "Machine learning is a method of data analysis that automates analytical model building. "
        "It is a branch of artificial intelligence based on the idea that systems can learn from data, "
        "identify patterns and make decisions with minimal human intervention. As big data continues to grow, "
        "machine learning will become even more important for extracting insights."
    ),
    (
        "Neural networks are a set of algorithms, modeled loosely after the human brain, that are designed "
        "to recognize patterns. They interpret sensory data through a kind of machine perception, labeling, "
        "or clustering of raw input. In practical terms, neural networks help cluster and classify data, and "
        "they are the foundation of deep learning algorithms."
    ),
]

print(f"Número de documentos: {len(documents)}")
for i, doc in enumerate(documents):
    print(f"Documento {i+1}: {len(doc.split())} palabras aproximadamente")

## 5) Función auxiliar de resumen abstractivo

In [ ]:
def summarize_abstractively(text: str, max_length: int = 80, min_length: int = 20, num_beams: int = 4) -> str:
    """Resume un texto usando el modelo abstractive BART.

    Args:
        text: Texto de entrada.
        max_length: Longitud máxima del resumen generada por el modelo.
        min_length: Longitud mínima del resumen.
        num_beams: Tamaño del beam search.

    Returns:
        Un string con el resumen generado.
    """
    result = summarizer(
        text,
        max_length=max_length,
        min_length=min_length,
        num_beams=num_beams,
        do_sample=False
    )
    return result[0]["summary_text"]

# Prueba rápida de la función
print("Resumen de prueba:")
print(summarize_abstractively(documents[0]))

## 6) Implementación de un resumen extractivo sencillo

In [ ]:
from nltk.tokenize import sent_tokenize

def split_sentences(text: str):
    """Divide el texto en frases usando NLTK."""
    return sent_tokenize(text)

def clean_and_tokenize(text: str):
    """Tokenización simple: minúsculas + eliminación de puntuación básica."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    return text.split()

def extractive_summary(text: str, ratio: float = 0.3) -> str:
    """Genera un resumen extractivo simple basado en frecuencia de palabras.

    Pasos:
    1. Divide el texto en frases.
    2. Calcula frecuencias de palabras en todo el documento.
    3. Asigna a cada frase una puntuación basada en la suma de frecuencias.
    4. Selecciona las frases mejor puntuadas según el `ratio`.
    5. Reconstruye el resumen respetando el orden original de las frases seleccionadas.
    """
    sentences = split_sentences(text)
    if len(sentences) == 0:
        return ""

    # Tokenizamos todo el texto para obtener frecuencias globales de palabras
    all_words = clean_and_tokenize(text)
    freqs = {}
    for w in all_words:
        freqs[w] = freqs.get(w, 0) + 1

    # Puntuamos cada frase
    sentence_scores = []
    for idx, sent in enumerate(sentences):
        words = clean_and_tokenize(sent)
        if not words:
            score = 0.0
        else:
            score = sum(freqs.get(w, 0) for w in words) / len(words)
        sentence_scores.append((idx, sent, score))

    # Determinamos cuántas frases vamos a conservar
    k = max(1, int(len(sentences) * ratio))

    # Seleccionamos las k frases con mayor puntuación
    sentence_scores_sorted = sorted(sentence_scores, key=lambda x: x[2], reverse=True)
    top_k = sentence_scores_sorted[:k]

    # Ordenamos las frases seleccionadas según el orden original
    top_k_sorted_by_index = sorted(top_k, key=lambda x: x[0])

    # Construimos el resumen
    summary_sentences = [sent for _, sent, _ in top_k_sorted_by_index]
    summary = " ".join(summary_sentences)
    return summary

# Prueba rápida de resumen extractivo
print("=== Resumen extractivo (doc 1) ===")
print(extractive_summary(documents[0], ratio=0.3))

## 7) Comparación cualitativa: extractivo vs abstractivo

In [ ]:
for i, doc in enumerate(documents):
    print("=" * 80)
    print(f"Documento {i+1} (primeras 50 palabras):")
    print(" ".join(doc.split()[:50]), "...")
    print()

    abs_sum = summarize_abstractively(doc, max_length=80, min_length=20, num_beams=4)
    ext_sum = extractive_summary(doc, ratio=0.4)

    print("--- Resumen abstractivo ---")
    print(abs_sum)
    print()
    print("--- Resumen extractivo ---")
    print(ext_sum)
    print()

## 8) Mini-métrica tipo ROUGE-1 (muy simplificada)

In [ ]:
def simple_rouge1(hyp: str, ref: str) -> float:
    """Métrica muy simple inspirada en ROUGE-1.

    Devuelve el porcentaje de palabras distintas del documento original (ref)
    que aparecen también en el resumen (hyp).
    """
    # Tokenización sencilla
    def tokenize(text):
        text = text.lower()
        text = re.sub(r"[^a-z0-9 ]+", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        if not text:
            return set()
        return set(text.split())

    ref_set = tokenize(ref)
    hyp_set = tokenize(hyp)

    if not ref_set:
        return 0.0

    overlap = len(ref_set & hyp_set)
    return overlap / len(ref_set)

# Prueba rápida
print(simple_rouge1("this is a test", "this is a simple test"))
print(simple_rouge1("completely different words", "this is a simple test"))

## 9) Evaluación básica: ROUGE-1 simple para todos los documentos

In [ ]:
results = []

for i, doc in enumerate(documents):
    abs_sum = summarize_abstractively(doc, max_length=80, min_length=20, num_beams=4)
    ext_sum = extractive_summary(doc, ratio=0.4)

    r_abs = simple_rouge1(abs_sum, doc)
    r_ext = simple_rouge1(ext_sum, doc)

    results.append((r_abs, r_ext))

    print(f"Documento {i+1}")
    print("ROUGE1 simple — abstractivo:", f"{r_abs:.3f}")
    print("ROUGE1 simple — extractivo:", f"{r_ext:.3f}")
    print("-" * 40)

results = np.array(results)
print("\nPromedio ROUGE1 simple — abstractivo:", results[:,0].mean())
print("Promedio ROUGE1 simple — extractivo:", results[:,1].mean())

## 10) Factualidad y alucinaciones en resumen abstractivo

En tus propios experimentos, es probable que observes algún caso donde:

- El modelo abstractivo introduce información que no aparece explícitamente en el texto original.
- Cambia ligeros matices (por ejemplo, refuerza o atenúa afirmaciones).
- Omite algún detalle numérico o técnico importante.

Estos casos ilustran el problema de la **factualidad** en resumen automático abstractivo:  
aunque el resultado parezca **fluido y bien redactado**, puede contener información **incorrecta o inventada**.

En un sistema real, esto puede ser peligroso (por ejemplo, en contextos médicos, legales o financieros).  
Por eso, las métricas puramente superficiales (como la cobertura léxica) no son suficientes: hace falta evaluación humana
y/o métricas específicas de factualidad.


## 11) Conclusiones

En este notebook hemos visto, de manera práctica:

- Cómo usar un modelo de resumen **abstractivo** basado en Transformers para generar resúmenes de textos en inglés.
- Cómo implementar un resumen **extractivo** sencillo usando frecuencias de palabras y selección de frases.
- Cómo comparar ambos enfoques en términos cualitativos (legibilidad, estilo, contenido) y cuantitativos
  mediante una mini-métrica tipo ROUGE-1 muy simplificada.
- Cómo el resumen abstractivo puede ser más fluido y condensado, pero también introducir alucinaciones,
  mientras que el extractivo es más fiel al texto original pero menos flexible.

Estas observaciones se conectan con los apartados **4.6–4.8** del Tema 4 (resumen automático), donde se discuten:
- Diferencias conceptuales y arquitecturales entre resumen extractivo y abstractivo.
- Métricas de evaluación como ROUGE y sus limitaciones.
- Problemas de factualidad y control en modelos generativos.

Como extensiones posibles, podrías:
- Probar otros modelos de resumen (por ejemplo, variantes de T5 o mBART).
- Trabajar con textos en español y ver hasta qué punto el modelo en inglés sigue siendo útil.
- Rediseñar la métrica para que tenga en cuenta también la longitud del resumen o la importancia de la información conservada.
